## ST207: Databases, Assessment II
### Question 2: Neo4j

**Jupyter Notebook Setup** (i.e. importing packages, libraries, and connection to my Neo4j AuraDB cluster)

In [65]:
import logging

from neo4j import GraphDatabase
from yfiles_jupyter_graphs import GraphWidget
from neo4j.exceptions import ServiceUnavailable

In [35]:
URI = "neo4j+s://dfe01aaf.databases.neo4j.io"
user = "neo4j"
password = "FeRrYEfsU-KMvMJ-SaPYxBpsuYeDTIw8KTHToVWzqxw"
AUTH = (user, password)

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()

**(a) Create a graph database, based on my Property Graph model translated from my E-R model.**

I have provided the Property Graph model I made in Neo4j, and one in Arrows.app, which shows all properties, attributes, and PK/FK relationships.

<div style="display: flex; justify-content: center; align-items: center;">
    <img src="fig/E-Commerce Property Graph Model.png" style="width:600px;height:400px;margin-right: 10px;">
    <img src="fig/E-Commerce Arrows.app Model.png" style="width:600px;height:400px;">
</div>

**(b) Populate your graph with the data you used in Assignment 1. This can be**

> **i) hardcoded (manually creating all nodes, labels, properties, and relationships), or**
>
> **ii) loaded from CSV files used in/exported from your first assignment.**

I followed **(ii)**, and loaded from my CSV files used in/exported from my first assignment. However, in gaining feedback as to how effective these were, I used Python code to edit these CSV files: joining some of them as I had introduced complexity (for example, I had 11 CSV files which I reduced to 8, which was a good idea for some of these tables existed as they were many-to-many relationships).

**Show the final (instance) graph. This can be**
> **i) on your Python code, using yFiles (see Seminar 9), or**
>
> **ii) a screenshot taken from your AuraDB instance and uploaded along with your code.**

I followed **(i)** and used yFiles.


In [75]:
with driver.session(database="neo4j") as session:
  graph = session.run("MATCH triple = (n)-[r]->(m) RETURN triple").graph()
  session.close()

GraphWidget(graph = graph)

/var/folders/sl/p4mhx_n92bd3278hvxdjbw_m0000gn/T/ipykernel_44215/3101377087.py:1: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session(database="neo4j") as session:


GraphWidget(layout=Layout(height='800px', width='100%'))

I have provided a screenshot from my AuraDB instance for a broad perspective of the data relationships and structure for completeness, following **(ii)**.

<div style="display: flex; justify-content: center; align-items: center;">
    <img src="fig/AuraDB Instance.png" style="width: 800px; height: 460px;">
</div>

**(c) Produce a list of all customers and their orders, including the list of products in each order and the (grand) total paid. Show the customer’s name and email, order number, order date, the list of products in each order and the total of each order.**

*NOTE: Order_Number, 70074, which returns Grand_Total as 0, as in Assignent 1 this order was randomly assigned with a Deduction/Promotion of 100%.*

In [50]:
records, summary, keys = driver.execute_query(
    '''
    MATCH (c)-[:PLACES_AN]->(o)-[:PURCHASES]->(p)
    RETURN 
      c.Name as Customer_Name, 
      c.Email_Address as Customer_Email, 
      o.Order_Number as Order_Number, 
      o.Date as Order_Date, 
      o.Grand_Total as Order_Total,
      // use of ChatGPT to collect Products into a dictionary
      COLLECT({Product_Number: p.Product_Number, Name: p.Product_Name}) AS Products 
    ''',
     database_="neo4j"
)

for r in records:
  print(r.data())

/var/folders/sl/p4mhx_n92bd3278hvxdjbw_m0000gn/T/ipykernel_44215/4161328966.py:1: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  records, summary, keys = driver.execute_query(


{'Customer_Name': 'Dylan Jackson', 'Customer_Email': 'vbowers@example.net', 'Order_Number': 69975, 'Order_Date': neo4j.time.DateTime(2023, 11, 26, 0, 0, 0, 0, tzinfo=<UTC>), 'Order_Total': 53690.0, 'Products': [{'Name': 'Beverage', 'Product_Number': 27}, {'Name': 'Cat Litter', 'Product_Number': 46}, {'Name': 'Educational Toy', 'Product_Number': 61}, {'Name': 'Bookcase', 'Product_Number': 66}]}
{'Customer_Name': 'Emily Crosby', 'Customer_Email': 'alison24@example.net', 'Order_Number': 50165, 'Order_Date': neo4j.time.DateTime(2024, 2, 27, 0, 0, 0, 0, tzinfo=<UTC>), 'Order_Total': 38805.0, 'Products': [{'Name': 'Pet Toy', 'Product_Number': 2}, {'Name': 'Beverage', 'Product_Number': 27}, {'Name': 'Drone', 'Product_Number': 44}, {'Name': 'Dog Food', 'Product_Number': 76}]}
{'Customer_Name': 'Emily Crosby', 'Customer_Email': 'alison24@example.net', 'Order_Number': 65435, 'Order_Date': neo4j.time.DateTime(2024, 7, 31, 0, 0, 0, 0, tzinfo=<UTC>), 'Order_Total': 200.0, 'Products': [{'Name': 'Sma

**(d) List all customers who have items in their baskets, so the company can make special offers based on their birthdays and any balance on existing gift cards. Retrieve the customer identification (e-mail), name, birthday, any balance from gift cards, and the list of products in their baskets.**

In [ ]:
records, summary, keys = driver.execute_query(
    '''
    MATCH (c)-[:HAS_A]->(b)-[:CONTAINS]->(p) // to retrieve only Customers with items in their Baskets
    OPTIONAL MATCH (c)-[:PAYS_BY]->(pay) // OPTIONAL MATCH is used as not all Customers have Payment Methods (i.e. no intention to buy)
    RETURN 
      c.Name as Customer_Name, 
      c.Email_Address as Customer_Email, 
      c.Date_of_Birth as Date_of_Birth, 
      pay.Current_Balance as Balance, 
      COLLECT({Product_Number: p.Product_Number, Name: p.Product_Name, Desired_Quantity: b.Desired_Quantity}) AS Products 
    ''',
     database_="neo4j"
)

for r in records:
  print(r.data())

/var/folders/sl/p4mhx_n92bd3278hvxdjbw_m0000gn/T/ipykernel_44215/1966810147.py:1: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  records, summary, keys = driver.execute_query(


{'Customer_Name': 'Dylan Jackson', 'Customer_Email': 'vbowers@example.net', 'Date_of_Birth': neo4j.time.DateTime(1926, 6, 17, 0, 0, 0, 0, tzinfo=<UTC>), 'Balance': None, 'Products': [{'Name': 'Ring', 'Desired_Quantity': 57, 'Product_Number': 1}, {'Name': 'Bakery Item', 'Desired_Quantity': 57, 'Product_Number': 5}, {'Name': 'Skincare Product', 'Desired_Quantity': 57, 'Product_Number': 6}, {'Name': 'Office Chair', 'Desired_Quantity': 57, 'Product_Number': 7}, {'Name': 'Pen', 'Desired_Quantity': 57, 'Product_Number': 8}, {'Name': 'Cat Litter', 'Desired_Quantity': 57, 'Product_Number': 12}, {'Name': 'Bed', 'Desired_Quantity': 57, 'Product_Number': 16}, {'Name': 'Supplement', 'Desired_Quantity': 57, 'Product_Number': 19}, {'Name': 'Diaper', 'Desired_Quantity': 57, 'Product_Number': 23}, {'Name': 'Headphone', 'Desired_Quantity': 57, 'Product_Number': 25}]}
{'Customer_Name': 'Emily Crosby', 'Customer_Email': 'alison24@example.net', 'Date_of_Birth': neo4j.time.DateTime(2005, 9, 18, 0, 0, 0, 0,

**(e) Identify the top two items sold in each product category, so the company can ensure that these products are kept in stock and marketed prominently. Retrieve the category names, the top two products from each category, and their total sales.**

I calculated the top two items sold in each product category as `Num_Ordered` for each unique `Product_Number`.

In [58]:
records, summary, keys = driver.execute_query(
    '''
    MATCH (c)-[:PLACES_AN]->(o)-[:PURCHASES]->(p)
    WITH 
      p.Category as Category,
      p.Product_Number as Product_Number,
      p.Product_Name as Product_Name,
      SUM(o.Num_Ordered) as Total_Sales
    ORDER BY Category, Total_Sales DESC
    WITH
      Category,
      COLLECT({Product_Number: Product_Number, Product_Name: Product_Name, Total_Sales: Total_Sales}) AS Products
    RETURN 
      Category,
      Products[0..2] as Top_Two_Sold_Items
      
    ''',
     database_="neo4j"
)

for r in records:
  print(r.data())

/var/folders/sl/p4mhx_n92bd3278hvxdjbw_m0000gn/T/ipykernel_44215/4144354584.py:1: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  records, summary, keys = driver.execute_query(


{'Category': 'Baby Products', 'Top_Two_Sold_Items': [{'Product_Name': 'Stroller', 'Total_Sales': 107, 'Product_Number': 52}, {'Product_Name': 'Diaper', 'Total_Sales': 98, 'Product_Number': 91}]}
{'Category': 'Books and Stationery', 'Top_Two_Sold_Items': [{'Product_Name': 'Pen', 'Total_Sales': 166, 'Product_Number': 71}, {'Product_Name': 'Pen', 'Total_Sales': 133, 'Product_Number': 8}]}
{'Category': 'Clothing and Apparel', 'Top_Two_Sold_Items': [{'Product_Name': 'Jacket', 'Total_Sales': 176, 'Product_Number': 82}, {'Product_Name': 'Accessory', 'Total_Sales': 156, 'Product_Number': 65}]}
{'Category': 'Electronics', 'Top_Two_Sold_Items': [{'Product_Name': 'Smartphone', 'Total_Sales': 275, 'Product_Number': 75}, {'Product_Name': 'Drone', 'Total_Sales': 182, 'Product_Number': 44}]}
{'Category': 'Furniture', 'Top_Two_Sold_Items': [{'Product_Name': 'Office Chair', 'Total_Sales': 237, 'Product_Number': 83}, {'Product_Name': 'Dining Table', 'Total_Sales': 144, 'Product_Number': 85}]}
{'Category

### Termination
We must close any widgets in use and the connection driver at the end.

In [ ]:
driver.close()

### A Statement on the Use of Generative AI Tools, As Per School and Course-Specific Policies
In **Question 2: Neo4j**, I used ChatGPT in the following instances:
- To collect information into dictionaries using `COLLECT({})`